In [5]:
from pyspark.sql import SparkSession

# 1. Aseguramos la sesión activa en el Kernel actual
spark = SparkSession.builder.appName("Storytelling_Final").getOrCreate()

# 2. Traemos los datos nativos persistidos en formato Parquet desde tu carpeta real
ruta_datos = "./Prueba"
df_historico = spark.read.parquet(ruta_datos)

# 3. Contamos cuántos registros totales logramos recuperar para el análisis
total_registros = df_historico.count()
print(f"ÉXITO: Conexión establecida. Hemos recuperado {total_registros} productos del histórico etiquetados con KMeans.")

ÉXITO: Conexión establecida. Hemos recuperado 5594 productos del histórico etiquetados con KMeans.


In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LeerCarpetaCompleta").getOrCreate()

# Lee TODOS los archivos CSV que estén dentro de esa carpeta y los junta automáticamente
df_historico = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("nombre_de_la_carpeta/*.csv")

print(f"¡Data unificada! Se cargaron {df_historico.count()} filas en total.")

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/Los AveMayo/nombre_de_la_carpeta/*.csv.

In [9]:
import os
os.listdir(".")

['.ipynb_checkpoints',
 'Conexión.ipynb',
 'kmeans_retail_v1',
 'Prueba',
 'Prueba ',
 'Prueba.ipynb',
 'S3_Scrapping_Dinamico.ipynb',
 'Scraper_Inicial.ipynb',
 'Semana 14',
 'Semana14.ipynb',
 'Semana15.ipynb',
 'Semana5.ipynb']

In [10]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Iniciamos la sesión de Spark
spark = SparkSession.builder \
    .appName("RetailDashboardData") \
    .getOrCreate()

# 2. CARGA INTELIGENTE: Intentamos leer la carpeta "Prueba" 
try:
    # Intentamos primero como CSV
    df_historico = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv("Prueba")
    print(f"¡Data unificada desde 'Prueba' (CSV)! Se cargaron {df_historico.count()} filas.")
except Exception:
    # Si falla, es porque es formato Parquet (muy común en Spark)
    df_historico = spark.read.parquet("Prueba")
    print(f"¡Data unificada desde 'Prueba' (Parquet)! Se cargaron {df_historico.count()} filas.")

# 3. Aplicamos la homologación de tus macro-categorías (Acuenta, Líder, etc.)
df_base = df_historico \
    .withColumn("supermercado", F.trim(F.translate(F.upper(F.col("supermercado")), "ÁÉÍÓÚÜ", "AEIOUU"))) \
    .withColumn("categoria", F.trim(F.translate(F.upper(F.col("categoria")), "ÁÉÍÓÚÜ", "AEIOUU"))) \
    .withColumn("marca", F.trim(F.regexp_replace(F.translate(F.upper(F.col("marca")), "ÁÉÍÓÚÜ", "AEIOUU"), "['\\.]", "")))

df_final_limpio = df_base.withColumn(
    "categoria_limpia",
    F.when(F.col("categoria").isin(
        "ACEITES", "ADEREZOS Y CONDIMENTOS", "ARROZ", "ARROZ, LEGUMBRES Y SEMILLAS", 
        "AZUCARES", "CONSERVAS", "CONSERVAS Y ENLATADOS", "DESPENSA", "DESPENSA GENERAL", 
        "PASTAS", "PASTAS FIDEOS Y SALSAS", "SALSAS", "LEGUMBRES", "HARINAS", "HARINAS LEVADURAS Y GRASAS"
    ), "DESPENSA Y ABARROTES")
    .when(F.col("categoria").isin(
        "LACTEOS", "LACTEOS Y CONGELADOS", "LACTEOS, HUEVOS Y REFRIGERADOS", 
        "LACTEOS/FIAMBRERIA", "LECHE EN POLVO", "LECHES LIQUIDAS Y CREMAS", 
        "MANTEQUILLAS Y MARGARINAS", "QUESOS", "YOGHURT Y POSTRES", "HUEVOS", "OTROS FRESCOS"
    ), "LACTEOS Y FRESCOS")
    .when(F.col("categoria").isin(
        "AVES", "CARNICERIA", "CERDO", "FIAMBRERIA", "FIAMBRERIA EMBUTIDOS Y QUESOS", 
        "FIAMBRERIA Y EMBUTIDOS", "PESCADOS Y MARISCOS"
    ), "CARNES Y FIAMBRERÍA")
    .when(F.col("categoria").isin(
        "AGUA CON GAS", "AGUA SIN GAS", "BEBIDAS", "BEBIDAS JUGOS Y AGUAS", "BEBIDAS LACTEAS Y VEGETALES"
    ), "BEBIDAS Y AGUAS")
    .when(F.col("categoria").isin("ASEO", "LIMPIEZA"), "LIMPIEZA Y ASEO")
    .otherwise("OTROS / ELABORADOS")
)

# 4. Exportamos a un Pandas CSV limpio ahí mismo en la carpeta de la Semana 15
df_final_limpio.toPandas().to_csv("datos_retail_dashboard.csv", index=False)
print("--> ¡Archivo 'datos_retail_dashboard.csv' generado con éxito para Streamlit! 🚀")

¡Data unificada desde 'Prueba' (CSV)! Se cargaron 912 filas.


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `supermercado` cannot be resolved. Did you mean one of the following? [`PAR1�Y�I҈ə<�  �`, `\	   GENÉRICA   TUCAPELTACUENTA   ACE   NETOSELECT0tCAROZZI   IANSA`].;
'Project [PAR1�Y�I҈ə<�  �#49, \	   GENÉRICA   TUCAPELTACUENTA   ACE   NETOSELECT0tCAROZZI   IANSA#50, trim(translate(upper('supermercado), ÁÉÍÓÚÜ, AEIOUU), None) AS supermercado#60]
+- Relation [PAR1�Y�I҈ə<�  �#49,\	   GENÉRICA   TUCAPELTACUENTA   ACE   NETOSELECT0tCAROZZI   IANSA#50] csv


In [11]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Aseguramos la sesión de Spark
spark = SparkSession.builder \
    .appName("RetailDashboardData") \
    .getOrCreate()

# 2. FORZAMOS LA LECTURA COMO PARQUET (Ya sabemos que no es CSV)
df_historico = spark.read.parquet("Prueba")
print(f"¡Data unificada correctamente desde Parquet! Se cargaron {df_historico.count()} filas.")

# Mostramos las columnas reales para confirmar que todo se abrió bonito
print("Columnas reales detectadas:", df_historico.columns)

# 3. Aplicamos la homologación de tus macro-categorías
df_base = df_historico \
    .withColumn("supermercado", F.trim(F.translate(F.upper(F.col("supermercado")), "ÁÉÍÓÚÜ", "AEIOUU"))) \
    .withColumn("categoria", F.trim(F.translate(F.upper(F.col("categoria")), "ÁÉÍÓÚÜ", "AEIOUU"))) \
    .withColumn("marca", F.trim(F.regexp_replace(F.translate(F.upper(F.col("marca")), "ÁÉÍÓÚÜ", "AEIOUU"), "['\\.]", "")))

df_final_limpio = df_base.withColumn(
    "categoria_limpia",
    F.when(F.col("categoria").isin(
        "ACEITES", "ADEREZOS Y CONDIMENTOS", "ARROZ", "ARROZ, LEGUMBRES Y SEMILLAS", 
        "AZUCARES", "CONSERVAS", "CONSERVAS Y ENLATADOS", "DESPENSA", "DESPENSA GENERAL", 
        "PASTAS", "PASTAS FIDEOS Y SALSAS", "SALSAS", "LEGUMBRES", "HARINAS", "HARINAS LEVADURAS Y GRASAS"
    ), "DESPENSA Y ABARROTES")
    .when(F.col("categoria").isin(
        "LACTEOS", "LACTEOS Y CONGELADOS", "LACTEOS, HUEVOS Y REFRIGERADOS", 
        "LACTEOS/FIAMBRERIA", "LECHE EN POLVO", "LECHES LIQUIDAS Y CREMAS", 
        "MANTEQUILLAS Y MARGARINAS", "QUESOS", "YOGHURT Y POSTRES", "HUEVOS", "OTROS FRESCOS"
    ), "LACTEOS Y FRESCOS")
    .when(F.col("categoria").isin(
        "AVES", "CARNICERIA", "CERDO", "FIAMBRERIA", "FIAMBRERIA EMBUTIDOS Y QUESOS", 
        "FIAMBRERIA Y EMBUTIDOS", "PESCADOS Y MARISCOS"
    ), "CARNES Y FIAMBRERÍA")
    .when(F.col("categoria").isin(
        "AGUA CON GAS", "AGUA SIN GAS", "BEBIDAS", "BEBIDAS JUGOS Y AGUAS", "BEBIDAS LACTEAS Y VEGETALES"
    ), "BEBIDAS Y AGUAS")
    .when(F.col("categoria").isin("ASEO", "LIMPIEZA"), "LIMPIEZA Y ASEO")
    .otherwise("OTROS / ELABORADOS")
)

# 4. Exportamos a un Pandas CSV limpio ahí mismo en la carpeta de la Semana 15
df_final_limpio.toPandas().to_csv("datos_retail_dashboard.csv", index=False)
print("--> ¡Archivo 'datos_retail_dashboard.csv' generado con éxito para Streamlit! 🚀")

¡Data unificada correctamente desde Parquet! Se cargaron 5594 filas.
Columnas reales detectadas: ['marca', 'supermercado', 'categoria', 'precio', 'supermercado_index', 'categoria_index', 'features', 'scaledFeatures', 'prediction']
--> ¡Archivo 'datos_retail_dashboard.csv' generado con éxito para Streamlit! 🚀


In [13]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Configuración de la página web
st.set_page_config(page_title="Dashboard Retail Ejecutivo", layout="wide")

st.title("📊 Cuadro de Mando Integral - Analítica de Retail")
st.markdown("---")

# 2. Carga de datos optimizada
@st.cache_data
def cargar_datos():
    return pd.read_csv("datos_retail_dashboard.csv")

df = cargar_datos()

# 3. Creación de las Pestañas (Tabs)
tab_est, tab_tac, tab_op = st.tabs([
    "📈 Nivel Estratégico (Vista Global)",
    "🎯 Nivel Táctico (Por Categorías)",
    "🛒 Nivel Operacional (Detalle Productos)"
])

Writing dashboard.py


In [15]:
import os
print(os.listdir('/home/jovyan/work/Los AveMayo/'))

['.ipynb_checkpoints', 'Conexión.ipynb', 'dashboard.py', 'datos_retail_dashboard.csv', 'kmeans_retail_v1', 'Prueba', 'Prueba ', 'Prueba.ipynb', 'S3_Scrapping_Dinamico.ipynb', 'Scraper_Inicial.ipynb', 'Semana 14', 'Semana14.ipynb', 'Semana15.ipynb', 'Semana5.ipynb']


In [16]:
!pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 3.4 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
